# LLM News Research Tool
## Notebook 2: Checking relevance with a language model

This notebook covers the second stage of the tool. A language model reads each article kept by the first stage: its title and a passage around the company mentions. It answers `direct`, `indirect` or `not`, with a certainty label (`sure` or `unsure`) and a one-line reason.

The main check is whether the model can identify irrelevant articles without removing relevant news. An article removed by this check never reaches the analyst, so missing relevant news is the biggest risk.

Every table is computed from the saved model answers and the checked labels. No model is called in this notebook.

In [1]:
import csv, json, os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
from src.config import DATA_DIR, EVAL_DIR
from src.fulltext import article_id
from src.fulltext_eval import RECORD_FILE, THRESHOLDS, _inputs, compare
from src.query import load_aliases

COMPANY = "Tata Motors"
record = json.loads((EVAL_DIR / RECORD_FILE).read_text(encoding="utf-8"))
aliases = load_aliases(DATA_DIR / "company_aliases.json")
pd.set_option("display.max_colwidth", 80)

## 1. The data

The saved record holds one answer for each of the 63 articles that the first stage keeps from the development searches. The answers come from Gemini 3.8 Flash. The labels follow labelling guide version 3. A blind re-check provides a second, checked set of labels. The table joins every answer to the checked labels by article id.

In [2]:
def labels_by_id(file_name):
    with open(EVAL_DIR / file_name, encoding="utf-8-sig", newline="") as handle:
        return {row["article_id"]: row["label"].strip().lower() for row in csv.DictReader(handle)}

checked = labels_by_id("v3_dev_tata_motors_checked.csv")
answers = record["records"]
found = [a for a in answers if a["article_id"] in checked]
unchanged = [a for a in found if checked[a["article_id"]] == a["label"]]
changed = [a for a in found if checked[a["article_id"]] != a["label"]]
missing = [a for a in answers if a["article_id"] not in checked]
pd.DataFrame([("Answers in the record", len(answers)), ("Found in the checked labels", len(found)),
              ("Label unchanged", len(unchanged)), ("Label changed", len(changed)),
              ("Missing from the checked labels", len(missing))], columns=["", "count"]).set_index("")

,count
,
Answers in the record,63
Found in the checked labels,63
Label unchanged,63
Label changed,0
Missing from the checked labels,0


The re-check changes none of the 63 labels.

## 2. The pass rule

Three measures decide whether the check can remove articles. Each has its own threshold, and all three are fixed before the run:

- **Relevant articles kept:** of the relevant articles kept by the first stage, the share the check also keeps.
- **Relevant stories kept:** a story is one article and its copies (the same cleaned link, or the same or nearly the same title). This is the share of all relevant stories in the labelled data, including stories the first stage misses, that still have at least one copy after the check. A story is narrower than a hand-grouped event, which can hold several different articles.
- **Not-relevant articles removed:** of the not-relevant articles kept by the first stage, the share the check removes.

For each measure, the required count is its threshold multiplied by its total, rounded up. A measure gets a pass or fail verdict only when its total reaches the minimum in the table. Below that minimum, there are too few items to judge.

In [3]:
MEASURES = {"recall_kept": "relevant articles kept", "stories": "relevant stories kept",
            "fp_removed": "not-relevant articles removed"}
pd.DataFrame([{"measure": MEASURES[name], "threshold": f"{fraction.numerator}/{fraction.denominator}",
               "minimum total for a verdict": minimum} for name, (fraction, minimum) in THRESHOLDS.items()]).set_index("measure")

,threshold,minimum total for a verdict
measure,,
relevant articles kept,43/45,20
relevant stories kept,41/43,20
not-relevant articles removed,2/3,8


## 3. Results

The rule is applied twice: first with the original labels, where the recomputed result must equal the saved record, and then with the checked labels.

In [4]:
articles, v3_labels, full_texts = _inputs("v3_dev_tata_motors.csv")
url_of = {article_id(a["url"]): a["url"] for a in articles}
saved_checks = {url_of[a["article_id"]]: {"label": a["model_label"], "certainty": a["certainty"],
                                          "reason": a["reason"], "source": "model"} for a in answers}
FULL_TEXT_HERE = len(full_texts) > 0

def acceptance_table(result):
    rows = {}
    for name, verdict in result["acceptance"].items():
        rows[MEASURES[name]] = {"kept or removed": verdict["count"], "total": verdict["total"],
                      "required": verdict["required"], "verdict": verdict["status"]}
    return pd.DataFrame(rows).T

if FULL_TEXT_HERE:
    again = compare(articles, v3_labels, full_texts, COMPANY, aliases, checks=saved_checks)
    assert again["acceptance"] == record["acceptance"], "the recomputed result differs from the saved record"
    print("With the original labels, the recomputed result equals the saved record.")
    display(acceptance_table(again))
else:
    print("The full texts are not on this computer; the saved record:")
    display(acceptance_table(record))

With the original labels, the recomputed result equals the saved record.


,kept or removed,total,required,verdict
relevant articles kept,42,45,43,fail
relevant stories kept,40,43,41,fail
not-relevant articles removed,17,18,12,pass


In [5]:
if FULL_TEXT_HERE:
    articles_c, checked_labels, _ = _inputs("v3_dev_tata_motors_checked.csv")
    now = compare(articles_c, checked_labels, full_texts, COMPANY, aliases, checks=saved_checks)
    print("With the checked labels:")
    display(acceptance_table(now))

With the checked labels:


,kept or removed,total,required,verdict
relevant articles kept,42,45,43,fail
relevant stories kept,40,42,41,fail
not-relevant articles removed,17,18,12,pass


In [6]:
if FULL_TEXT_HERE:
    from src.evaluate import is_relevant
    from src.fulltext_eval import stories
    from src.preprocess import normalize_url

    unique = {}
    for article in articles:
        unique.setdefault(normalize_url(article.get("url") or ""), article)
    versions = {"original": ({normalize_url(u): l for u, l in v3_labels.items()}, again),
                "checked": ({normalize_url(u): l for u, l in checked_labels.items()}, now)}
    original, checked_all = versions["original"][0], versions["checked"][0]
    print(f"Labelled articles: {len(unique)}")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame([{"title": unique[k]["title"], "original label": original[k],
                               "checked label": checked_all[k],
                               "kept by Stage 1": article_id(k) in again["full"]["kept_ids"]}
                              for k in unique if original[k] != checked_all[k]]))
        lost = []
        for name, (labels_now, result) in versions.items():
            for group in stories(unique):
                if any(is_relevant(labels_now[k]) for k in group) and                         not any(article_id(k) in result["full+check"]["kept_ids"] for k in group):
                    stage1 = any(article_id(k) in result["full"]["kept_ids"] for k in group)
                    lost.append({"labels": name, "lost story": unique[sorted(group)[0]]["title"],
                                 "lost by": "the check" if stage1 else "Stage 1"})
        display(pd.DataFrame(lost))

Labelled articles: 104


,title,original label,checked label,kept by Stage 1
0,Return of the world car: China backs one-size-fits-all approach,indirect,not,False


,labels,lost story,lost by
0,original,"Used Truck Market to Reach USD 70.74 Billion by 2031 as Digital Auctions and Fleet Renewal Reshape the Secondary Market, Reports Mordor Intelligence",the check
1,original,"Piyush Goyal says India will become global car manufacturing hub, European firms to follow",the check
2,original,Return of the world car: China backs one-size-fits-all approach,Stage 1
3,checked,"Used Truck Market to Reach USD 70.74 Billion by 2031 as Digital Auctions and Fleet Renewal Reshape the Secondary Market, Reports Mordor Intelligence",the check
4,checked,"Piyush Goyal says India will become global car manufacturing hub, European firms to follow",the check


Only the story total changes, from 43 to 42. One article outside the 63 candidates, "Return of the world car", changes from `indirect` to `not` in the re-check. The first stage does not keep it, so with the original labels its story is lost by the first stage, not by the check. Under both label versions, the check loses the same two stories, and every verdict stays the same.

The check removes 17 of the 18 articles that are not relevant, which is what it is for. But it also removes relevant articles, above the allowed limit: one measure passes and two fail.

## 4. Relevant articles the check removes

In [7]:
dropped = [a for a in answers if a["decision"] == "dropped" and checked.get(a["article_id"]) in ("direct", "indirect")]
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([{"title": a["title"], "human label": checked[a["article_id"]],
                           "model label": a["model_label"], "certainty": a["certainty"],
                           "model's reason": a["reason"]} for a in dropped]))

,title,human label,model label,certainty,model's reason
0,Piyush Goyal sees India becoming global car hub as Europe’s regulations hurt competitiveness,indirect,not,sure,Mentions Tata Motors only through a historical anecdote about its past acquisition of Jaguar.
1,"Used Truck Market to Reach USD 70.74 Billion by 2031 as Digital Auctions and Fleet Renewal Reshape the Secondary Market, Reports Mordor Intelligence",indirect,not,sure,It is an industry report on the used truck market that mentions Tata Motors only in a bulleted list without substantive business information.
2,"Piyush Goyal says India will become global car manufacturing hub, European firms to follow",indirect,not,sure,Tata Motors is mentioned only in passing through a minister's historical anecdote about its past acquisition of Jaguar.


All three are `indirect` industry news, not announcements by the company: two on India as a car-making hub, and one used-truck market report. The model answers `not` for each, so it is too strict on indirect industry news. Each of these wrong answers is marked `sure`, so the certainty label cannot pick out articles that are safe to remove.

## 5. Ranking instead of filtering

Removing 17 articles that are not relevant does not make up for losing relevant ones: the analyst never sees a removed article and cannot know it is missing. So the check does not remove articles. Its answers go to the next stage as a possible ranking signal instead. A wrong answer may move an article lower in the list, but it never removes the article.

## 6. Checking the record

These assertions confirm that the saved record is complete and consistent, so the results above rest on valid data.

In [8]:
LABELS, CERTAINTIES, DECISIONS = {"direct", "indirect", "not"}, {"sure", "unsure"}, {"kept", "dropped", "borderline"}
assert len(answers) == 63
assert len({a["article_id"] for a in answers}) == len(answers), "an article appears twice"
assert all(a["model_label"] in LABELS for a in answers), "an answer without a valid label"
assert all(a["certainty"] in CERTAINTIES for a in answers), "an answer without a valid certainty"
assert all((a["reason"] or "").strip() for a in answers), "an answer without a reason"
assert all(a["decision"] in DECISIONS for a in answers)
assert all(a["article_id"] in url_of for a in answers), "an answer that does not join to an article"
assert not missing, "an answer missing from the checked labels"
loaded = [name for name in ("groq", "langchain_groq", "langchain_google_genai", "google.genai") if name in sys.modules]
assert not loaded, f"a model provider library is loaded: {loaded}"
print(f"{len(answers)} answers: every label, certainty, reason and article id is valid, and each article appears once.")
print("No model provider library is loaded, so no model can be called.")
print("Decisions:", {d: sum(a['decision'] == d for a in answers) for d in sorted(DECISIONS)})

63 answers: every label, certainty, reason and article id is valid, and each article appears once.
No model provider library is loaded, so no model can be called.
Decisions: {'borderline': 0, 'dropped': 20, 'kept': 43}


## 7. Limitations

- The development set is small: 63 candidates from three search days.
- The answers come from one run of one model. They show that this model, used as a filter, loses relevant indirect news; they say nothing about other models or about ranking.